# 10. Gamma Sharpe Ratio (GSR) — POC v0

**Goal.** Empirical "Sharpe for gamma" on a Uni v3 pool: how many fees the pool pays per unit of gamma risk absorbed, with dispersion around that mean.

**Three quantities (per pool × day)**:

1. **σ*_pool** — pool-implied vol from yield equilibrium.
   `σ*² = 8 · F / TVL_FR`, where `F` = fees over the window, `TVL_FR = 2·L_active·√P` is the full-range equivalent of the active concentrated liquidity.

2. **σ_realized** — Garman-Klass on hourly OHLC, annualized.

3. **CV[IL]** = `Std[IL] / E[|IL|]` — coefficient of variation of the daily IL cost.
   `E[|IL|] = L·√P·σ²·Δt / 4`. `Std[IL]` heuristic via realized quarticity (`Σ r_i⁴`).

**Edge** = σ*_pool − σ_realized. Positive = LP over-paid.

**Pool note.** The user spec referenced ETH/USDC **0.3%** mainnet, but our indexed universe is USDC/WETH **0.05%** (`0x88e6…5640`). Math is identical; concentration leverage will be different (0.05% pool is tighter, so leverage typically higher).

**Data window.** 90 days, 2026-01-06 → 2026-04-06 (cap = full mart coverage).


In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8124")
os.environ.setdefault("CLICKHOUSE_DATABASE", "analytics")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "1234")

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from core.db import ClickHouseDB, ClickHouseSettings
db = ClickHouseDB.from_settings(ClickHouseSettings())

POOL       = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"  # USDC/WETH 0.05%
START_DATE = "2026-01-06"
END_DATE   = "2026-04-06"
print(f"Pool:   {POOL}")
print(f"Window: {START_DATE} → {END_DATE}")

# Auto-cleanup: close CH connection on kernel shutdown so sockets don't pile up
import atexit as _atexit
_atexit.register(lambda: db.close() if 'db' in globals() and db is not None else None)


Pool:   0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640
Window: 2026-01-06 → 2026-04-06


<function __main__.<lambda>()>

## §1 — Load `py_pool_gsr_daily`

In [2]:
gsr = db.read_query(f"""
    SELECT *
    FROM py_pool_gsr_daily
    WHERE lower(pool_address) = lower('{POOL}')
      AND date BETWEEN toDate('{START_DATE}') AND toDate('{END_DATE}')
    ORDER BY date
""")
print(f"rows: {gsr.height}")
gsr.head(5)


rows: 91


date,pool_address,V_usd,L_active,sqrt_P,TVL_FR_usd,fees_daily_usd,fees_7d_usd,sigma_star_pool_daily_annual,sigma_star_pool_7d_annual,sigma_realized_annual,edge,E_abs_IL,std_IL,cv_IL,n_hours,n_returns
date,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
2026-01-06,"""0x88e6a0c2ddd26feeb64f039a2c41…",2.0686e7,1.7785e18,57.417954,2.0423e8,19592.713737,19592.713737,0.529268,0.200044,0.539856,-0.339812,20900.849088,8103.212529,0.387698,24,23
2026-01-07,"""0x88e6a0c2ddd26feeb64f039a2c41…",2.0880e7,1.9302e18,56.268088,2.1722e8,22939.59283,42532.306567,0.555311,0.285795,0.424201,-0.138407,16106.577717,8360.036093,0.519045,24,23
2026-01-08,"""0x88e6a0c2ddd26feeb64f039a2c41…",2.0321e7,1.8273e18,55.726983,2.0366e8,18330.95364,60863.260207,0.512658,0.353072,0.437025,-0.083952,15052.10676,4836.002333,0.321284,24,23
2026-01-09,"""0x88e6a0c2ddd26feeb64f039a2c41…",2.0265e7,1.8281e18,55.534167,2.0304e8,16732.247998,77595.508205,0.49054,0.39927,0.535502,-0.136232,13313.219762,4619.780333,0.347007,24,23
2026-01-10,"""0x88e6a0c2ddd26feeb64f039a2c41…",2.1119e7,4.0358e18,55.53307,4.4824e8,6168.008605,83763.51681,0.20045,0.279198,0.193274,0.085925,2539.704011,726.680609,0.286128,24,23


## §2 — Sanity checks (printed)

In [3]:
# Sanity 1: daily volume (back out from fees and fee_tier)
fee_tier = 0.05 / 100  # 0.05% as ratio (= 0.0005)
gsr_with_volume = gsr.with_columns(
    (pl.col("fees_daily_usd") / fee_tier).alias("est_volume_daily_usd"),
)
avg_vol = float(gsr_with_volume["est_volume_daily_usd"].mean())
print(f"Sanity 1 — avg daily volume (back-out from fees / fee_tier): ${avg_vol:,.0f}")
print(f"            check vs Uniswap UI for USDC/WETH 0.05% (Q1 2026 typical range $100M-$300M)")

# Sanity 2: TVL_FR / active TVL = concentration leverage
gsr_with_lev = gsr.with_columns(
    (pl.col("TVL_FR_usd") / pl.col("V_usd")).alias("concentration_leverage"),
)
mean_lev = float(gsr_with_lev["concentration_leverage"].mean())
median_lev = float(gsr_with_lev["concentration_leverage"].median())
print(f"\nSanity 2 — concentration leverage (TVL_FR / active TVL):")
print(f"            mean   = {mean_lev:.1f}×")
print(f"            median = {median_lev:.1f}×")
print(f"            target: 5-30× for USDC/WETH (the 0.05% pool is on the tight end)")

# Sanity 3: σ_realized annualized
mean_sr = float(gsr["sigma_realized_annual"].mean())
median_sr = float(gsr["sigma_realized_annual"].median())
min_sr = float(gsr["sigma_realized_annual"].min())
max_sr = float(gsr["sigma_realized_annual"].max())
print(f"\nSanity 3 — σ_realized annualized:")
print(f"            mean   = {mean_sr:.1%}")
print(f"            median = {median_sr:.1%}")
print(f"            range  = {min_sr:.1%} → {max_sr:.1%}")
print(f"            target: 50-80% for ETH recently")

# Sanity 4: σ*_pool same order of magnitude
mean_ss = float(gsr["sigma_star_pool_7d_annual"].mean())
median_ss = float(gsr["sigma_star_pool_7d_annual"].median())
print(f"\nSanity 4 — σ*_pool (7d window) annualized:")
print(f"            mean   = {mean_ss:.1%}")
print(f"            median = {median_ss:.1%}")
print(f"            target: same order as σ_realized if pool near equilibrium")

# Edge stats
print(f"\nEdge = σ*_pool_7d − σ_realized:")
print(f"  mean   = {float(gsr['edge'].mean()):+.1%}")
print(f"  median = {float(gsr['edge'].median()):+.1%}")
print(f"  days_positive = {gsr.filter(pl.col('edge') > 0).height}/{gsr.height}")

# CV[IL]
print(f"\nCV[IL] (sqrt(Σr⁴) / Σr²):")
print(f"  mean   = {float(gsr['cv_IL'].mean()):.3f}")
print(f"  median = {float(gsr['cv_IL'].median()):.3f}")
print(f"  baseline √(3/24) ≈ 0.354 for iid Gaussian hourly returns")


Sanity 1 — avg daily volume (back-out from fees / fee_tier): $186,562,163
            check vs Uniswap UI for USDC/WETH 0.05% (Q1 2026 typical range $100M-$300M)

Sanity 2 — concentration leverage (TVL_FR / active TVL):
            mean   = 15.3×
            median = 16.5×
            target: 5-30× for USDC/WETH (the 0.05% pool is on the tight end)

Sanity 3 — σ_realized annualized:
            mean   = 70.8%
            median = 57.1%
            range  = 15.2% → 351.0%
            target: 50-80% for ETH recently

Sanity 4 — σ*_pool (7d window) annualized:
            mean   = 60.2%
            median = 51.6%
            target: same order as σ_realized if pool near equilibrium

Edge = σ*_pool_7d − σ_realized:
  mean   = -10.5%
  median = -3.6%
  days_positive = 40/91

CV[IL] (sqrt(Σr⁴) / Σr²):
  mean   = 0.433
  median = 0.409
  baseline √(3/24) ≈ 0.354 for iid Gaussian hourly returns


## §3 — Plot: σ*_pool vs σ_realized (90d)

In [4]:
days = gsr["date"].to_list()
fig = go.Figure()
fig.add_trace(go.Scatter(x=days, y=gsr["sigma_star_pool_daily_annual"].to_list(),
                          name="σ*_pool (daily)", line=dict(color="#9ca3af", width=1, dash="dot")))
fig.add_trace(go.Scatter(x=days, y=gsr["sigma_star_pool_7d_annual"].to_list(),
                          name="σ*_pool (7d, primary)", line=dict(color="#22c55e", width=2)))
fig.add_trace(go.Scatter(x=days, y=gsr["sigma_realized_annual"].to_list(),
                          name="σ_realized (Garman-Klass)", line=dict(color="#ef4444", width=2)))
fig.update_layout(
    height=420, title_text="σ*_pool vs σ_realized — annualized (90d)",
    yaxis_title="annualized vol", yaxis_tickformat=".0%",
)
fig.show()


## §4 — Plot: edge (σ*_pool − σ_realized) with zero line

In [5]:
edge = gsr["edge"].to_list()
colors = ["#22c55e" if e is not None and e > 0 else "#ef4444" for e in edge]
fig = go.Figure()
fig.add_trace(go.Bar(x=days, y=edge, marker_color=colors, name="edge"))
fig.add_hline(y=0, line_dash="dash", line_color="white")
fig.update_layout(
    height=320, title_text="edge = σ*_pool (7d) − σ_realized (90d)",
    yaxis_title="edge (pp annualized)", yaxis_tickformat=".0%",
)
fig.show()


## §5 — Plot: CV[IL] (90d)

In [6]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=days, y=gsr["cv_IL"].to_list(), name="CV[IL]", line=dict(color="#3b82f6", width=2)))
fig.add_hline(y=0.354, line_dash="dot", line_color="white",
              annotation_text="baseline √(3/24) ≈ 0.354")
fig.update_layout(height=320, title_text="CV[IL] = √(Σr⁴) / Σr² (daily, 90d)", yaxis_title="CV[IL]")
fig.show()


## §6 — Combined view (PM use case)

In [7]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
    subplot_titles=("σ*_pool (7d) vs σ_realized", "edge", "CV[IL]"))
fig.add_trace(go.Scatter(x=days, y=gsr["sigma_star_pool_7d_annual"].to_list(),
                          name="σ*_pool 7d", line=dict(color="#22c55e")), row=1, col=1)
fig.add_trace(go.Scatter(x=days, y=gsr["sigma_realized_annual"].to_list(),
                          name="σ_realized", line=dict(color="#ef4444")), row=1, col=1)
fig.add_trace(go.Bar(x=days, y=gsr["edge"].to_list(),
                      marker_color=["#22c55e" if e is not None and e > 0 else "#ef4444" for e in gsr["edge"].to_list()],
                      name="edge"), row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="white", row=2, col=1)
fig.add_trace(go.Scatter(x=days, y=gsr["cv_IL"].to_list(),
                          name="CV[IL]", line=dict(color="#3b82f6")), row=3, col=1)
fig.add_hline(y=0.354, line_dash="dot", line_color="white", row=3, col=1)
fig.update_yaxes(tickformat=".0%", row=1, col=1)
fig.update_yaxes(tickformat=".0%", row=2, col=1)
fig.update_layout(height=720, showlegend=True, title_text="Gamma Sharpe Ratio — full PM view (90d)")
fig.show()


## §7 — Sample tail (last 15 days)

In [8]:
display_df = gsr.tail(15).select(
    "date",
    pl.col("V_usd").round(0).alias("active_TVL"),
    pl.col("TVL_FR_usd").round(0).alias("TVL_FR"),
    pl.col("sigma_star_pool_7d_annual").round(4).alias("σ*_pool_7d"),
    pl.col("sigma_realized_annual").round(4).alias("σ_realized"),
    pl.col("edge").round(4).alias("edge"),
    pl.col("E_abs_IL").round(0).alias("E[|IL|]"),
    pl.col("cv_IL").round(3).alias("CV[IL]"),
)
display_df


date,active_TVL,TVL_FR,σ*_pool_7d,σ_realized,edge,E[|IL|],CV[IL]
date,f64,f64,f64,f64,f64,f64,f64
2026-03-23,6.7204789e7,1.0930e9,0.4899,0.9467,-0.4568,441982.0,0.565
2026-03-24,6.7217454e7,1.1099e9,0.5062,0.4689,0.0373,99070.0,0.433
2026-03-25,6.7639108e7,1.1304e9,0.5008,0.4171,0.0838,33864.0,0.303
2026-03-26,6.9924496e7,1.1798e9,0.4814,0.5782,-0.0968,114692.0,0.497
2026-03-27,7.4752596e7,1.2841e9,0.4655,0.5713,-0.1058,140665.0,0.683
…,…,…,…,…,…,…,…
2026-04-02,7.3168918e7,1.2479e9,0.4963,0.4931,0.0032,214749.0,0.548
2026-04-03,7.3637095e7,1.2535e9,0.4746,0.2958,0.1788,16259.0,0.413
2026-04-04,7.3648416e7,1.2566e9,0.4647,0.2014,0.2634,10548.0,0.409


## §8 — Findings & caveats

- **Edge slightly negative** (mean ≈ −10.5pp) over 90d : the pool's fee yield does **not** fully compensate the realized vol absorbed.
  - Consistent with the 22.7× LVR/theoretical ratio observed in the LVR notebook (concentration amplification).
- **Concentration leverage = 15.3×** (median): typical for the 0.05% pool. For 0.3% pool you'd expect 5-10×.
- **CV[IL] ≈ 0.43**, slightly above the iid Gaussian baseline 0.354 → fat-tailed hourly returns on this pool.
- **σ*_pool_daily** is much noisier than 7d. Use 7d as primary signal.

**Caveats**:
- "Std[IL] = (V_FR/2) × √(Σr⁴) / 4" is a heuristic; refine with proper Andersen-Shephard `√(2/3 × Σr⁴)` if needed (constant factor 0.816).
- E[|IL|] uses full-range V_FR equivalent — the **realized** IL on actual concentrated positions is amplified by the concentration leverage. The σ*_pool/σ_realized comparison is invariant to this: both measure the pool's fair vol.
- Garman-Klass uses hourly OHLC. For thin-volume hours, GK can be biased; could complement with realized variance from `mart_price_analytics_daily.realized_vol_*`.

**Next steps if results pass review**:
1. Promote `py_pool_gsr_daily` to a versioned dbt mart.
2. Index the 0.3% pool and re-run side by side (compare concentration leverages).
3. Add `Var[edge]` rolling 30d → "Gamma Sharpe Ratio" proper = `mean(edge) / std(edge)`.
4. Wire to PM dashboard Block 1 (regime/carry environment).
